# Class Activity 2: OOP Fundamentals & Refactoring the CLI Task Manager

Refactor the procedural Task Manager into two classes:
- **`Task`** – one task (`id`, `description`, `completed`) with `mark_complete()`, `to_dict()`, `__str__`, `__repr__`
- **`TaskManager`** – holds a list of `Task` objects and handles add / list / complete / delete plus JSON save & load

Run the cells from top to bottom (**Runtime → Run all**).

## Step 1: Create folders

In [1]:
!mkdir -p data src
!rm -f data/tasks.json   # start fresh each run

## Step 2: `src/task.py` — the `Task` class

In [2]:
%%writefile src/task.py
# my-oop-task-manager/src/task.py


class Task:
    """
    Represents a single task in the task manager.
    """

    def __init__(self, id, description, completed=False):
        """
        Initializes a new Task object.

        Args:
            id (int): A unique identifier for the task.
            description (str): A brief description of the task.
            completed (bool, optional): The completion status of the task. Defaults to False.
        """
        self.id = id
        self.description = description
        self.completed = completed

    def mark_complete(self):
        """Marks the task as completed."""
        self.completed = True

    def to_dict(self):
        """Converts the Task object to a dictionary for JSON serialization."""
        return {
            "id": self.id,
            "description": self.description,
            "completed": self.completed,
        }

    def __str__(self):
        """Returns a human-readable string representation of the task."""
        status = "Completed" if self.completed else "Pending"
        return f"ID: {self.id} | Description: {self.description} | Status: {status}"

    def __repr__(self):
        """Returns an official string representation of the task object for debugging."""
        return f"Task(id={self.id}, description='{self.description}', completed={self.completed})"


Writing src/task.py


## Step 3: `src/task_manager.py` — the `TaskManager` class

In [3]:
%%writefile src/task_manager.py
# my-oop-task-manager/src/task_manager.py
import json
import os

from .task import Task  # Import the Task class from the same package


class TaskManager:
    """
    Manages the collection of Task objects, including loading, saving,
    and performing operations like add, list, complete, and delete.
    """

    def __init__(self, data_file="data/tasks.json"):
        """
        Initializes the TaskManager.
        Loads tasks from the specified data file or starts with an empty list.
        """
        self.data_file = data_file
        # Resolve relative paths against the project root (one level above src/)
        if os.path.isabs(data_file):
            self._data_file_path = data_file
        else:
            project_root = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
            self._data_file_path = os.path.normpath(os.path.join(project_root, data_file))
        self.tasks = self._load_tasks()
        self.next_id = self._get_next_task_id()

    def _get_next_task_id(self):
        """Generates the next unique ID for a new task."""
        if not self.tasks:
            return 1
        return max(task.id for task in self.tasks) + 1

    def _load_tasks(self):
        """
        Internal method to load tasks from the JSON file.
        Converts dictionaries from JSON back into Task objects.
        """
        if not os.path.exists(self._data_file_path):
            os.makedirs(os.path.dirname(self._data_file_path), exist_ok=True)
            return []
        try:
            with open(self._data_file_path, "r", encoding="utf-8") as f:
                raw_tasks = json.load(f)
            # Convert raw dictionaries back into Task objects
            return [Task(t["id"], t["description"], t["completed"]) for t in raw_tasks]
        except json.JSONDecodeError:
            print("Warning: tasks.json is empty or corrupted. Starting with an empty task list.")
            return []
        except FileNotFoundError:
            return []

    def _save_tasks(self):
        """
        Internal method to save tasks to the JSON file.
        Converts Task objects to dictionaries before saving.
        """
        tasks_as_dicts = [task.to_dict() for task in self.tasks]
        os.makedirs(os.path.dirname(self._data_file_path), exist_ok=True)
        with open(self._data_file_path, "w", encoding="utf-8") as f:
            json.dump(tasks_as_dicts, f, indent=4, ensure_ascii=False)

    def add_task(self, description):
        """Adds a new task."""
        new_task = Task(self.next_id, description)
        self.tasks.append(new_task)
        self.next_id += 1  # Increment for the next task
        self._save_tasks()
        print(f"Task '{description}' added with ID {new_task.id}.")
        return new_task

    def list_tasks(self):
        """Lists all current tasks."""
        if not self.tasks:
            print("No tasks found.")
            return
        print("\n--- Your Tasks ---")
        for task in self.tasks:
            print(task)  # Task.__str__ is called automatically
        print("------------------")

    def find_task(self, task_id):
        """Returns the Task with the given ID, or None if not found."""
        for task in self.tasks:
            if task.id == task_id:
                return task
        return None

    def complete_task(self, task_id):
        """Marks a specific task as completed."""
        task = self.find_task(task_id)
        if task is None:
            print(f"Error: Task with ID {task_id} not found.")
            return False
        if task.completed:
            print(f"Task ID {task_id} is already completed.")
            return False
        task.mark_complete()  # Use the Task object's method
        self._save_tasks()
        print(f"Task ID {task_id} marked as completed.")
        return True

    def delete_task(self, task_id):
        """Deletes a task by its ID."""
        original_len = len(self.tasks)
        self.tasks = [task for task in self.tasks if task.id != task_id]
        if len(self.tasks) < original_len:
            self._save_tasks()
            print(f"Task ID {task_id} deleted successfully.")
            return True
        print(f"Error: Task with ID {task_id} not found.")
        return False


Writing src/task_manager.py


## Step 4: `src/__init__.py` — makes `src` a package

In [4]:
%%writefile src/__init__.py
# This file can be empty. It indicates that the 'src' directory is a Python package.


Writing src/__init__.py


## Step 5: Import the classes

In [5]:
import os, sys
sys.path.insert(0, os.getcwd())

from src.task import Task
from src.task_manager import TaskManager

## Step 6: Demo — Task objects, `__str__` vs `__repr__`

In [6]:
t = Task(99, 'Demo task')
print(str(t))
print(repr(t))
t.mark_complete()
print(t)
print(t.to_dict())

ID: 99 | Description: Demo task | Status: Pending
Task(id=99, description='Demo task', completed=False)
ID: 99 | Description: Demo task | Status: Completed
{'id': 99, 'description': 'Demo task', 'completed': True}


## Step 7: Result — show all tasks (1 PENDING, 1 COMPLETED)

In [7]:
manager = TaskManager()

manager.add_task('Task 1')
manager.add_task('Task 2')
manager.complete_task(2)

manager.list_tasks()

Task 'Task 1' added with ID 1.
Task 'Task 2' added with ID 2.
Task ID 2 marked as completed.

--- Your Tasks ---
ID: 1 | Description: Task 1 | Status: Pending
ID: 2 | Description: Task 2 | Status: Completed
------------------


### Saved JSON (Task objects → dicts via `to_dict()`)

In [8]:
!cat data/tasks.json

[
    {
        "id": 1,
        "description": "Task 1",
        "completed": false
    },
    {
        "id": 2,
        "description": "Task 2",
        "completed": true
    }
]

### Reload from JSON (dicts → Task objects)

In [9]:
reloaded = TaskManager()
print(reloaded.tasks)
print('next_id =', reloaded.next_id)

[Task(id=1, description='Task 1', completed=False), Task(id=2, description='Task 2', completed=True)]
next_id = 3


## Step 8 (optional): Interactive CLI
Run this cell to use the menu. Choose **5** to exit.

In [10]:
def display_menu():
    print("\n--- OOP Task Manager Menu ---")
    print("1. Add Task")
    print("2. List Tasks")
    print("3. Complete Task")
    print("4. Delete Task")
    print("5. Exit")
    print("-------------------------")


def main_colab():
    manager = TaskManager()
    while True:
        display_menu()
        choice = input("Enter your choice (1-5): ").strip()
        if choice == "1":
            description = input("Enter task description: ").strip()
            if description:
                manager.add_task(description)
            else:
                print("Task description cannot be empty.")
        elif choice == "2":
            manager.list_tasks()
        elif choice in ("3", "4"):
            try:
                task_id = int(input("Enter task ID: ").strip())
            except ValueError:
                print("Invalid input. Please enter a number for Task ID.")
                continue
            if choice == "3":
                manager.complete_task(task_id)
            else:
                manager.delete_task(task_id)
        elif choice == "5":
            print("Exiting Task Manager. Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 5.")

# main_colab()   # <- remove the # to start the interactive menu